
# Everstock media engine

**To start: Runtime → Run all.** That's it. The engine boots ComfyUI + LTX-2.5, then watches the shared
`everstock-media/jobs/` folder and renders whatever lands there into `everstock-media/outputs/`.
It shuts itself down after 20 idle minutes (or 3 hours), so it never bills while nobody is using it.

Queue jobs by asking Claude ("make a 5s vertical clip of…"), or by dropping a job file into `jobs/` by hand.

First time on an account: L4 GPU is preset; if Colab asks to allow `HF_TOKEN_LTX` or Google Drive access, approve it.

## 1 · GPU + settings

In [ ]:
import subprocess, time, os
SESSION_START = time.time()
CU_PER_HOUR = {"L4": 1.71, "A100": 5.40, "T4": 1.19}; USD_PER_CU = 0.10
IDLE_MINUTES = 20        # auto-stop after this many minutes with no jobs
MAX_SESSION_HOURS = 3    # hard cap per session
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
GPU_KEY = next((k for k in CU_PER_HOUR if k in gpu), None)
print("GPU:", gpu or "NONE: Runtime > Change runtime type > L4")
assert GPU_KEY in ("L4", "A100"), "Needs an L4 or A100 GPU (Runtime > Change runtime type)"

## 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MEDIA = "/content/drive/MyDrive/everstock-media"
MODELS = "/content/drive/MyDrive/everstock_comfy/models"   # private weights cache (~46 GB)
assert os.path.isdir(MEDIA), "everstock-media folder not found in this account's My Drive"

In [ ]:
# 2b · Brand switch: which brand's Drive folder the engine reads jobs from and renders into.
# Engine code stays in everstock-media/_engine. Originals are never edited; this runs re-pointed copies.
# ShopLink needs a My Drive shortcut to the shared "shoplink-media" folder on this account.
import os, re, pathlib, subprocess

BRAND = "shoplink"            # "everstock" or "shoplink"

ROOTS = {
    "everstock": "/content/drive/MyDrive/everstock-media",
    "shoplink":  "/content/drive/MyDrive/shoplink-media",
}
OLD    = ROOTS["everstock"]               # path the engine scripts were written against
ENGINE = f"{OLD}/_engine"                 # engine code lives here for every brand
MEDIA  = ROOTS[BRAND]

assert os.path.isdir(ENGINE), f"Engine not found at {ENGINE}"
assert os.path.isdir(MEDIA), (
    f"{MEDIA} not found. In Drive, open the shared 'shoplink-media' folder → "
    "Organize → Add shortcut → My Drive, then rerun cell 2 and this cell."
)
for d in ("brand", "ads", "jobs", "status", "outputs", "outputs/finals", "outputs/stills"):
    os.makedirs(f"{MEDIA}/{d}", exist_ok=True)

# Bring the engine up to date first, so the copies below are of the latest worker.
print(subprocess.run(["python3", f"{ENGINE}/upgrade_v9.py"], capture_output=True, text=True).stdout.strip())

# Re-pointed copies of the engine scripts: brand paths → MEDIA, _engine paths stay put.
RUN = "/content/engine_run"
os.makedirs(RUN, exist_ok=True)
KEEP = "\x00ENGINE\x00"
for name in ("worker.py", "beats_to_jobs.py", "check_finals.py", "make_cut.py"):
    src_path = pathlib.Path(ENGINE, name)
    if not src_path.exists():
        print(f"skip {name} (not in _engine)")
        continue
    src = src_path.read_text()
    out = src.replace(ENGINE, KEEP).replace(OLD, MEDIA).replace(KEEP, ENGINE)
    out = out.replace('os.path.join(MEDIA, "_engine")', repr(ENGINE))  # engine folder never moves
    pathlib.Path(RUN, name).write_text(out)
    left = len(re.findall(r"everstock-media(?!/_engine)", out)) if BRAND != "everstock" else 0
    print(f"{name}: {src.count(OLD)} path(s) re-pointed"
          + (f"  ⚠ {left} other 'everstock-media' reference(s) left, check by hand" if left else ""))

SYS_DIR = ENGINE  # the worker reads its templates from the engine folder, whatever the brand

# ComfyUI writes into /content/ComfyUI/output; point it at this brand's folder.
out_dir = "/content/ComfyUI/output"
if os.path.islink(out_dir):
    os.unlink(out_dir)
if os.path.isdir("/content/ComfyUI") and not os.path.exists(out_dir):
    os.symlink(MEDIA, out_dir)

print(f"\nBrand: {BRAND}\nJobs in:     {MEDIA}/jobs\nRenders to:  {MEDIA}/outputs")

<!-- STRIPPED on import to eyelore/legacy-engine: this code cell embedded the V2 design system as a ~750 KB base64 JSON blob (100 files, 560033 bytes decoded) and wrote it to _engine/v2ds/ on Drive. Removed to keep the notebook small; the original is on Drive. -->

## 3 · Install ComfyUI

In [ ]:
%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || (cd ComfyUI && git pull -q)
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q -U "huggingface_hub[hf_xet]"

## 4 · Model weights (cached in Drive after the first run)

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
FILES = [
  ("Lightricks/LTX-2.5", "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors", "diffusion_models"),
  ("Lightricks/LTX-2.5", "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors", "text_encoders"),
  ("Comfy-Org/gemma-4",  "text_encoders/gemma4_e2b_it_bf16.safetensors", "text_encoders"),
  ("Lightricks/LTX-2.5", "vae/ltx-2.5-video-vae-bf16.safetensors", "vae"),
  ("Lightricks/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors", "vae"),
  ("Lightricks/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", "latent_upscale_models"),
]
token = None
for repo, path, sub in FILES:
    name = os.path.basename(path); src = os.path.join(MODELS, sub, name)
    os.makedirs(os.path.dirname(src), exist_ok=True)
    if not os.path.exists(src):
        token = token or userdata.get("HF_TOKEN_LTX")
        print("download", name)
        tmp = hf_hub_download(repo, path, token=token, local_dir="/content/hf_tmp")
        !cp "{tmp}" "{src}" && rm -f "{tmp}"
    dst = f"/content/ComfyUI/models/{sub}/{name}"; os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst): os.symlink(src, dst)
print("weights ready")

## 5 · Workflows + output folder

In [ ]:
wf = "/content/ComfyUI/user/default/workflows"; os.makedirs(wf, exist_ok=True)
base = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates"
for t in ["video_ltx2_5_t2v", "video_ltx2_5_i2v", "video_ltx2_5_flf2v"]:
    !wget -q -O "{wf}/{t}.json" "{base}/{t}.json"
out = "/content/ComfyUI/output"
if os.path.isdir(out) and not os.path.islink(out):
    !rm -rf "{out}"
if not os.path.exists(out): os.symlink(MEDIA, out)
print("ready")

## 6 · Run the engine
Watches `jobs/`, renders, writes `outputs/` + `status/`, and stops the runtime when idle. Interrupt (■) to stop the engine but keep the runtime.

In [ ]:
MAX_SESSION_HOURS = 6
SYS_DIR = ENGINE
if BRAND == "shoplink":
    # The feeder queues faces → scene stills → clips → final cut as each stage's inputs land.
    # Which ads: --plan file (_engine/shoplink_case_run.json = travel jewelry case; ring feeder is shoplink_jobs.py). Progress: shoplink-media/status/_shoplink_feeder.json
    import subprocess
    feeder = subprocess.Popen(["python3", f"{ENGINE}/shoplink_jobs_v2.py", "--media", MEDIA, "--run", RUN, "--plan", f"{ENGINE}/shoplink_case_run.json", "--watch"],
                              stdout=open("/content/shoplink_feeder.log", "a"), stderr=subprocess.STDOUT)
    time.sleep(10)
    print(open("/content/shoplink_feeder.log").read()[-800:])
else:
    AD = "b-ttny-first-warehouse"
    !python3 {RUN}/beats_to_jobs.py {AD}
%run -i {RUN}/worker.py